# Bee Orientation Estimation

Trains and evaluates CNNs that predict the orientation $\theta$ of a bee
centered in an image patch (produced by `preprocessing.py`).

**Key idea:** instead of regressing $\theta$ directly (which is discontinuous
at the $\pm\pi$ wrap-around), the model outputs two continuous values
$(\sin\theta, \cos\theta)$. The orientation is reconstructed via
$\hat\theta = \operatorname{atan2}(\widehat{\sin\theta}, \widehat{\cos\theta})$.

- **Loss:** cosine loss $1 - \cos(\Delta\theta)$ — the raw output is
  L2-normalized onto the unit circle and compared to the unit target,
  which equals the MSE of the normalized $(\sin, \cos)$ pair (up to a
  factor of 2) and depends *only* on the angular error:
  $\approx \Delta\theta^2/2$ for small errors, flattening
  (outlier-robust) towards $180°$.
- **Error metric:** signed angular error
  $\Delta\theta = \operatorname{atan2}(\sin(\theta - \hat\theta), \cos(\theta - \hat\theta))$,
  reported as mean/median absolute error in degrees, plus threshold accuracies
  and an axial (mod-180°) companion metric.
- **Models:** ResNet-18, ResNet-50, MobileNetV4 (all via `timm`).

**Head/tail ambiguity:** if the tracker's `orientation_angle` comes from an
axial estimator (e.g. an ellipse fit), labels do not distinguish head from
tail and are arbitrarily flipped by 180°. Training $(\sin\theta, \cos\theta)$
against such labels is unlearnable noise (MAE saturates near 90°). In that
case set `Config.axial_labels = True`: the model then regresses
$(\sin 2\theta, \cos 2\theta)$ and decodes with $\operatorname{atan2}(\cdot)/2$.

In [1]:
# Uncomment on first run if dependencies are missing on the HPC node
# (ipywidgets is what renders the animated tqdm progress bars in Jupyter):
# %pip install --user timm torch torchvision pandas matplotlib tqdm ipywidgets

In [ ]:
from __future__ import annotations

import random
from dataclasses import replace

import torch

# All the reusable logic lives in orientation.py (a real module, so both this
# notebook and train.py share one source of truth and DataLoader workers can
# import it). Restart the kernel after editing orientation.py or bee_dataset.py.
from orientation import (
    Config,
    DEVICE as device,
    RunResult,
    build_comparison_table,
    create_model,
    load_or_build_index,
    make_loaders,
    plot_error_diagnostics,
    plot_label_check,
    plot_predictions,
    plot_training_curves,
    seed_everything,
    split_samples,
    train_and_evaluate,
)

## Configuration

Everything tunable lives in a single `Config` dataclass.

In [ ]:
cfg = Config()
print(f"Device: {device}")

seed_everything(cfg.seed)

## Sample index

Each trajectory file `rec1_trajectories/<traj_id>.txt` holds rows
`frame_nb, position_x, position_y, object_class, orientation_angle`.
A sample pairs the crop `crops/<traj_id>/frame_<frame>.png` with its
orientation label.

Border crops are filtered **geometrically**: `preprocessing.py` clips a crop
exactly when the bee position is within `half_size` of the frame border, so
partial crops can be identified from the trajectory positions alone — no
image needs to be opened (per-file I/O is what makes indexing slow on a
scratch filesystem). A spot check on a few random crops verifies the rule;
existence is checked with one directory listing per trajectory.

In [ ]:
samples = load_or_build_index(cfg)

## Train / val / test split

Default splits **by trajectory**: consecutive frames of the same bee are nearly
identical, so a random frame-level split would leak information from train to test.
Note the trajectory counts printed below — with small fractions the test set may
contain only a handful of bees, and the test metrics carry that variance.

In [ ]:
train_samples, val_samples, test_samples = split_samples(samples, cfg)
for split_name, split in (("train", train_samples), ("val", val_samples), ("test", test_samples)):
    n_traj = len({s.traj_id for s in split})
    print(f"{split_name}: {len(split)} samples from {n_traj} trajectories")

## Dataset & dataloaders

`Sample` and `BeeOrientationDataset` are defined in **`bee_dataset.py`**, not
here: with `num_workers > 0` each DataLoader worker unpickles the dataset in a
fresh process, and (under Python ≥3.14's default `forkserver` start method on
Linux) that process re-imports the class from its defining module — classes
defined in notebook cells live in `__main__` and can't be found, which crashes
worker startup. After editing `bee_dataset.py`, restart the kernel to pick up
the changes.

Targets are `[sin(k·theta), cos(k·theta)]` with `k = 2` for axial labels and
`k = 1` otherwise. This order is assumed everywhere, including
`atan2(pred[:, 0], pred[:, 1]) / k` at decode time.

In [ ]:
train_loader, val_loader, test_loader = make_loaders(
    train_samples, val_samples, test_samples, cfg
)

## ⚠️ Gate: verify the label convention before training

**Confirmed convention** (checked visually against the crops):
`orientation_angle` is a compass bearing in **degrees** — 0° = up,
90° = right, clockwise on screen. The index converts it to display radians
(`angles_in_degrees = True`, `angle_convention = "compass"`), so the arrows
below are drawn from the exact values used as training targets and must lie
along the bees' bodies. Rerun this cell after any change to the label
pipeline (delete `sample_index.csv` first — the cache stores converted angles).

**Head/tail check (observed):** arrows mostly point at the head, with
occasional 180° flips — so the labels are *directional* and
`axial_labels = False` is correct (truly axial labels would flip ~50/50).
The flipped minority acts as outlier noise, which the bounded cosine loss
tolerates by design. After training, the error histogram below tells the
flip rate: it is the mass of the secondary mode at ±180°. Only if that mode
turns out large (roughly >20–30%) reconsider `Config.axial_labels = True`.

In [ ]:
# n=24 gives a rough feel for the head/tail flip rate; bump for a better estimate.
# save_path=None renders inline; pass a path to also write a PNG.
plot_label_check(train_samples, cfg, n=24)

## Models

All three architectures come from `timm` with the classification head replaced
by a 2-unit linear layer predicting `[sin(k·theta), cos(k·theta)]`.

## Loss & metrics

- **Loss** — `cosine_loss`: `1 - cosine_similarity(output, target)`, i.e. the
  prediction is L2-normalized and compared to the unit target. Equivalent to
  `1 - cos(k·Δθ)`, a pure function of the angular error (`F.cosine_similarity`
  handles the near-zero-norm case with its built-in eps). One caveat: the
  gradient `sin(Δ)` vanishes at exactly 180°, so predictions pointing
  backwards get weak gradients early in training — the price of bounded,
  outlier-robust loss for flipped labels.
- `mae_deg` / `median_ae_deg` / `rmse_deg`: primary error, wrap-around-safe.
  In axial mode these are computed mod 180°.
- `axial_mae_deg`: error mod 180°, always reported. Comparing it with
  `mae_deg` separates "wrong axis" from "right axis but flipped 180°".
- `acc15/30/45_deg`: percentage of predictions within 15°/30°/45°.
- `loss`: the cosine loss itself — interpretable via `1 - cos(Δθ)`
  (e.g. 0.1 ≈ 26° error), but lead with the degree-space metrics.

## Baselines

Context for the comparison table: a uniform-random predictor scores 90° MAE
(45° axial), and predicting the circular mean of the train labels is the
honest floor any model must beat.

## Training & evaluation loops

Per-epoch history and final test metrics are written to
`checkpoints/{name}_history.csv` / `{name}_test_metrics.json` as they are
produced, so nothing is lost if the kernel dies mid-run on a shared node.

## Tiny-subset overfit check

Trains ResNet-18 on 128 samples and evaluates on those **same** samples — a
pure memorization test of the pipeline, run once before spending hours on
full training. **Pass:** loss falls well below 0.05 and MAE below ~10° within
the 60 epochs. **Fail:** loss stalls near 1.0 / MAE near 90°, meaning labels,
targets, decode, or loss are inconsistent somewhere — do not start full
training. (Says nothing about generalization; checkpoints go to a separate
`checkpoints_overfit/` dir, which can be deleted afterwards.)

In [14]:
overfit_cfg = replace(
    cfg,
    epochs=60,
    batch_size=32,
    num_workers=0,
    rotation_augment=False,
    checkpoint_dir=cfg.checkpoint_dir.parent / "checkpoints_overfit",
)
tiny = random.Random(cfg.seed).sample(train_samples, 128)
tiny_train, tiny_val, tiny_test = make_loaders(tiny, tiny, tiny, overfit_cfg)
overfit_result = train_and_evaluate("resnet18", overfit_cfg, tiny_train, tiny_val, tiny_test)


=== resnet18 ===


Using 4 GPUs with DataParallel!


resnet18:   0%|          | 0/60 [00:00<?, ?epoch/s]

epoch 1/60 · train:   0%|          | 0/4 [00:00<?, ?batch/s]

epoch 1/60 · val:   0%|          | 0/4 [00:00<?, ?batch/s]

...

epoch 60/60 · train:   0%|          | 0/4 [00:00<?, ?batch/s]

epoch 60/60 · val:   0%|          | 0/4 [00:00<?, ?batch/s]

epoch  60 | train loss 0.0126 | val loss 0.0048 | val MAE   4.24°


resnet18 · test:   0%|          | 0/4 [00:00<?, ?batch/s]

test: loss 0.0048 | MAE 4.16° | median 3.19°


## Run all experiments

**Do not run until the sanity-check gate above passes** — training three
models on labels with the wrong units, convention, or head/tail assumption
costs hours of HPC time and produces meaningless numbers.

In [ ]:
results: dict[str, RunResult] = {}
for name in cfg.model_names:
    results[name] = train_and_evaluate(name, cfg, train_loader, val_loader, test_loader)


=== resnet18 ===
Using 4 GPUs with DataParallel!


resnet18:   0%|          | 0/15 [00:00<?, ?epoch/s]

epoch 1/15 · train:   0%|          | 0/11089 [00:24<?, ?batch/s]

epoch 1/15 · val:   0%|          | 0/2325 [00:15<?, ?batch/s]

epoch   1 | train loss 0.0986 | val loss 0.0982 | val MAE  13.07°  *


...

epoch   4 | train loss 0.0622 | val loss 0.0991 | val MAE  12.91°


epoch 5/15 · train:   0%|          | 0/11089 [00:00<?, ?batch/s]

epoch 5/15 · val:   0%|          | 0/2325 [00:00<?, ?batch/s]

epoch   5 | train loss 0.0576 | val loss 0.0836 | val MAE  11.19°  *


epoch 6/15 · train:   0%|          | 0/11089 [00:00<?, ?batch/s]

## Model comparison

Degree-space metrics are the ones to read; `loss` is the cosine loss
`1 - cos(k·Δθ)` on the test set. The baseline rows give the context: models
must clearly beat the circular-mean floor, and an MAE near the random
baseline (90°, or 45° axial) suggests head/tail-ambiguous labels.

In [ ]:
comparison = build_comparison_table(results, train_samples, test_samples, cfg)
comparison.round(2)

In [ ]:
plot_training_curves(results)

## Error diagnostics

Histogram: secondary modes at ±180° indicate head/tail flips. Scatter of
error vs. true angle: reveals systematic bias near particular orientations
that the histogram hides.

In [ ]:
plot_error_diagnostics(results, cfg)

## Qualitative results

Red arrow: ground truth. Blue arrow: prediction of the best model (lowest test MAE).

In [ ]:
best_name = comparison.loc[list(results)]["mae_deg"].idxmin()
best_res = results[best_name]
print(f"Best model: {best_name}")

best_model = create_model(best_name, cfg).to(device)
best_model.load_state_dict(
    torch.load(best_res.checkpoint_path, map_location=device, weights_only=True)
)
best_model.eval()

plot_predictions(test_samples, best_model, cfg)